<a href="https://colab.research.google.com/github/trang1981/ELAPS/blob/main/VIB_Dwindow_Refit_20Seeds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# VIB — D_WINDOW: 20 STRATIFIED SPLITS AND A/B REFITS
#
# A: Fixed first 60 days
# B: Event anchored
#
# Cùng cohort, nhãn và train/test trong mỗi lần chia.
# 18 đặc trưng gốc; XGBoost; không lấy mẫu.
# Seed chia dữ liệu: 1000–1019.
# Seed mô hình: cố định 42.
#
# Kết quả thực được hiển thị ngay bên dưới ô code.
# ============================================================


# ============================================================
# 1. IMPORTS VÀ GOOGLE DRIVE
# ============================================================

import gc
import json
import time

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version

import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

try:
    from xgboost import XGBClassifier
except ModuleNotFoundError:
    raise RuntimeError(
        "Chưa có xgboost. Chạy một ô: !pip install xgboost "
        "rồi chạy lại ô này."
    )

drive.mount("/content/drive", force_remount=False)


# ============================================================
# 2. CẤU HÌNH
# ============================================================

DW_ROOT = Path("/content/drive/MyDrive/Fintect")

DW_A_FILE = (
    DW_ROOT
    / "final_landmark_60d"
    / "final_dataset_landmark_60d_no_auto_job.csv"
)

DW_B_FILE = DW_ROOT / "VIB_FLDC_60D.csv"

DW_ID_COL = "CUSTOMER_NUMBER"
DW_TARGET_A = "TARGET_60D"
DW_TARGET_B = "COUNT_CREDITCARD"

DW_EXPECTED_CUSTOMERS = 94453
DW_EXPECTED_POSITIVES = 4374
DW_EXPECTED_TRAIN = 75562
DW_EXPECTED_TEST = 18891

DW_R = 20
DW_TEST_SIZE = 0.20

# Định trước danh sách seed, không chọn seed theo kết quả.
DW_SPLIT_SEEDS = list(range(1000, 1020))

# Seed mô hình giữ nguyên cấu hình gốc.
DW_MODEL_SEED = 42

DW_FEATURES = [
    "CLIENT_SEX",
    "EB_REGISTER_CHANNEL",
    "SMS",
    "VERIFY_METHOD",
    "AGE",
    "LOGIN_PER_ACTIVE_DAY",
    "INTEREST_RATE_RATIO",
    "TRANS_LV1_MODE",
    "TRANS_LV2_MODE",
    "TRANS_AMOUNT_MAX",
    "TRANS_AMOUNT_MEAN",
    "COUNT_CA_ACCT",
    "AVG_CA_BALANCE",
    "COUNT_TD_ACCT",
    "AVG_TD_BALANCE",
    "COUNT_OF_LOAN",
    "AVG_LOAN_AMOUNT",
    "TOTAL_LOAN_AMOUNT"
]

DW_XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "logloss",
    "objective": "binary:logistic",
    "random_state": DW_MODEL_SEED,
    "n_jobs": -1
}

if not DW_ROOT.is_dir():
    raise FileNotFoundError(f"Không truy cập được: {DW_ROOT}")

if len(DW_FEATURES) != 18 or len(set(DW_FEATURES)) != 18:
    raise RuntimeError("Cần đúng 18 đặc trưng không trùng.")

if "TENURE_AT_CUTOFF" in DW_FEATURES:
    raise RuntimeError("A/B không sử dụng TENURE_AT_CUTOFF.")

if len(DW_SPLIT_SEEDS) != DW_R:
    raise RuntimeError("Số seed không bằng R.")

if len(set(DW_SPLIT_SEEDS)) != DW_R:
    raise RuntimeError("Danh sách seed bị trùng.")


# ============================================================
# 3. HÀM ĐỌC VÀ KIỂM TRA DỮ LIỆU
# ============================================================

def dw_standardize_id(series):
    """Chuẩn hóa ID theo cách trong mã gốc."""

    result = (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    invalid_values = ["", "NAN", "<NA>", "NONE", "NULL"]

    return result.mask(
        result.str.upper().isin(invalid_values),
        pd.NA
    )


def dw_load_branch(file_path, target_col, branch_name):
    """
    Đọc bảng đặc trưng đã tạo.
    Nếu dữ liệu bất thường, dừng để kiểm tra;
    không tự đổi nhãn hoặc loại khách hàng.
    """

    if not file_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy: {file_path}")

    frame = pd.read_csv(file_path, low_memory=False)

    frame.columns = (
        frame.columns.astype(str)
        .str.strip()
        .str.upper()
    )

    if frame.columns.duplicated().any():
        raise RuntimeError(
            f"Nhánh {branch_name}: tên cột bị trùng sau chuẩn hóa."
        )

    required = [DW_ID_COL, target_col] + DW_FEATURES
    missing = [col for col in required if col not in frame.columns]

    if missing:
        raise RuntimeError(
            f"Nhánh {branch_name} thiếu cột: {missing}"
        )

    frame[DW_ID_COL] = dw_standardize_id(frame[DW_ID_COL])

    if frame[DW_ID_COL].isna().any():
        raise RuntimeError(
            f"Nhánh {branch_name}: có ID thiếu hoặc không hợp lệ."
        )

    if frame[DW_ID_COL].duplicated().any():
        raise RuntimeError(
            f"Nhánh {branch_name}: có khách hàng trùng ID."
        )

    labels = pd.to_numeric(frame[target_col], errors="coerce")

    # Kiểm tra nhãn trước khi ép kiểu, tránh cắt phần thập phân.
    if labels.isna().any() or not labels.isin([0, 1]).all():
        raise RuntimeError(
            f"Nhánh {branch_name}: nhãn gốc không phải 0/1 đầy đủ. "
            "Cần rà soát dữ liệu, không tự chuyển nhãn."
        )

    if "TARGET_SHARED" in frame.columns:
        existing = pd.to_numeric(
            frame["TARGET_SHARED"],
            errors="coerce"
        )

        if not np.array_equal(
            existing.to_numpy(),
            labels.to_numpy()
        ):
            raise RuntimeError(
                f"Nhánh {branch_name}: TARGET_SHARED khác nhãn gốc."
            )

    frame["TARGET_SHARED"] = labels.astype(int)

    if len(frame) != DW_EXPECTED_CUSTOMERS:
        raise RuntimeError(
            f"Nhánh {branch_name}: có {len(frame):,} khách hàng, "
            f"khác {DW_EXPECTED_CUSTOMERS:,} trong lần chạy gốc."
        )

    positives = int(frame["TARGET_SHARED"].sum())

    if positives != DW_EXPECTED_POSITIVES:
        raise RuntimeError(
            f"Nhánh {branch_name}: có {positives:,} ca dương, "
            f"khác {DW_EXPECTED_POSITIVES:,} trong lần chạy gốc."
        )

    print(f"\nLOAD {branch_name}")
    print("File:", file_path)
    print("Shape:", frame.shape)
    print("Customers:", f"{len(frame):,}")
    print("Positives:", f"{positives:,}")

    return frame


# ============================================================
# 4. PREPROCESSING VÀ MÔ HÌNH
# Giữ các lựa chọn trong mã gốc chị gửi.
# ============================================================

def dw_build_preprocessor(X_train):

    numeric_cols = (
        X_train.select_dtypes(include=np.number)
        .columns.tolist()
    )

    categorical_cols = (
        X_train.select_dtypes(exclude=np.number)
        .columns.tolist()
    )

    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            )
        )
    ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="MISSING"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols)
    ])


def dw_get_model():
    """Tạo mô hình mới cho mỗi nhánh ở mỗi lần chia."""

    return XGBClassifier(**DW_XGB_PARAMS)


# ============================================================
# 5. LOAD VÀ GHÉP A/B THEO ID
# ============================================================

DW_DF_A = dw_load_branch(DW_A_FILE, DW_TARGET_A, "A")
DW_DF_B = dw_load_branch(DW_B_FILE, DW_TARGET_B, "B")

DW_A = DW_DF_A.set_index(DW_ID_COL, drop=False)
DW_B = DW_DF_B.set_index(DW_ID_COL, drop=False)

if set(DW_A.index) != set(DW_B.index):
    raise RuntimeError(
        "A/B không có cùng tập khách hàng. "
        "Không tự lấy giao hai tập."
    )

# B được sắp theo đúng thứ tự khách hàng của A.
DW_B = DW_B.loc[DW_A.index].copy()

DW_Y = DW_A["TARGET_SHARED"].astype(int)

DW_TARGET_MISMATCH = int(
    (
        DW_Y.to_numpy()
        != DW_B["TARGET_SHARED"].to_numpy()
    ).sum()
)

if DW_TARGET_MISMATCH != 0:
    raise RuntimeError(
        f"Có {DW_TARGET_MISMATCH} khách hàng mang nhãn khác nhau."
    )

DW_X_A = DW_A[DW_FEATURES].copy()
DW_X_B = DW_B[DW_FEATURES].copy()

print("\n===== KIỂM TRA A/B =====")
print("Khách hàng A:", f"{len(DW_X_A):,}")
print("Khách hàng B:", f"{len(DW_X_B):,}")
print("Ca dương:", f"{int(DW_Y.sum()):,}")
print("Target mismatch:", DW_TARGET_MISMATCH)
print("Cùng thứ tự ID:", DW_X_A.index.equals(DW_X_B.index))
print("Đặc trưng mỗi nhánh:", len(DW_FEATURES))
print("Seed mô hình:", DW_MODEL_SEED)
print("Seed chia dữ liệu:", DW_SPLIT_SEEDS)
print("Sampling: None")


# ============================================================
# 6. THƯ MỤC KẾT QUẢ RIÊNG
# Mỗi lần chạy toàn bộ ô tạo một thư mục mới.
# ============================================================

DW_OUTPUT_ROOT = DW_ROOT / "VIB_DWINDOW_REFIT_20SEEDS"

DW_TIMESTAMP = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

DW_RUN_DIR = DW_OUTPUT_ROOT / f"run_{DW_TIMESTAMP}"
DW_RUN_DIR.mkdir(parents=True, exist_ok=False)

DW_METADATA = {
    "status": "running",
    "R_requested": DW_R,
    "completed_splits": 0,
    "n_customers": len(DW_Y),
    "positives": int(DW_Y.sum()),
    "source_A": str(DW_A_FILE),
    "source_B": str(DW_B_FILE),
    "features": DW_FEATURES,
    "split_seeds": DW_SPLIT_SEEDS,
    "test_size": DW_TEST_SIZE,
    "stratified": True,
    "sampling": "None",
    "model_parameters": DW_XGB_PARAMS,
    "versions": {
        package: version(package)
        for package in [
            "numpy",
            "pandas",
            "scikit-learn",
            "xgboost"
        ]
    },
    "percentile_interpretation": (
        "Empirical percentiles across repeated splits; "
        "not a confidence interval of the mean."
    )
}


def dw_save_metadata():
    (DW_RUN_DIR / "run_manifest.json").write_text(
        json.dumps(
            DW_METADATA,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


dw_save_metadata()

print("\nThư mục kết quả:", DW_RUN_DIR)
print("Phiên bản thư viện:", DW_METADATA["versions"])


# ============================================================
# 7. CHẠY 20 LẦN CHIA VÀ HUẤN LUYỆN LẠI A/B
# ============================================================

DW_POSITIONS = np.arange(len(DW_Y))
DW_ROWS = []
DW_TOTAL_START = time.perf_counter()

try:
    for run_number, split_seed in enumerate(
        DW_SPLIT_SEEDS,
        start=1
    ):
        split_start = time.perf_counter()

        # Một phân hoạch chung cho A và B.
        train_idx, test_idx = train_test_split(
            DW_POSITIONS,
            test_size=DW_TEST_SIZE,
            random_state=split_seed,
            stratify=DW_Y.to_numpy()
        )

        if (
            len(train_idx) != DW_EXPECTED_TRAIN
            or len(test_idx) != DW_EXPECTED_TEST
        ):
            raise RuntimeError(
                f"Seed {split_seed}: số train/test không đúng."
            )

        if np.intersect1d(train_idx, test_idx).size != 0:
            raise RuntimeError("Train/test bị chồng lặp.")

        y_train = DW_Y.iloc[train_idx].to_numpy()
        y_test = DW_Y.iloc[test_idx].to_numpy()

        if len(np.unique(y_train)) != 2:
            raise RuntimeError("Train thiếu một lớp.")

        if len(np.unique(y_test)) != 2:
            raise RuntimeError("Test thiếu một lớp.")

        test_ids = DW_Y.index[test_idx].to_numpy()

        # Lưu ID, nhãn và thứ tự train/test.
        ordered_idx = np.concatenate([
            train_idx,
            test_idx
        ])

        split_table = pd.DataFrame({
            "CUSTOMER_NUMBER": DW_Y.index[ordered_idx],
            "TARGET": DW_Y.iloc[ordered_idx].to_numpy(),
            "SPLIT": (
                ["TRAIN"] * len(train_idx)
                + ["TEST"] * len(test_idx)
            ),
            "ORDER_IN_PARTITION": np.concatenate([
                np.arange(len(train_idx)),
                np.arange(len(test_idx))
            ])
        })

        split_table.to_csv(
            DW_RUN_DIR / f"split_seed_{split_seed}.csv",
            index=False
        )

        print(
            f"\n===== LẦN {run_number}/{DW_R} "
            f"— SPLIT SEED {split_seed} =====",
            flush=True
        )

        print(
            f"Train={len(train_idx):,}; "
            f"Test={len(test_idx):,}; "
            f"Positive train={int(y_train.sum()):,}; "
            f"Positive test={int(y_test.sum()):,}",
            flush=True
        )

        branch_auc = {}
        branch_prob = {}

        for branch, X in [
            ("A", DW_X_A),
            ("B", DW_X_B)
        ]:
            print(f"Đang fit nhánh {branch}...", flush=True)

            X_train = (
                X.iloc[train_idx]
                .copy()
                .reset_index(drop=True)
            )

            X_test = (
                X.iloc[test_idx]
                .copy()
                .reset_index(drop=True)
            )

            # Preprocessing mới cho nhánh/lần chia này.
            # Chỉ fit trên train.
            preprocessor = dw_build_preprocessor(X_train)

            X_train_processed = preprocessor.fit_transform(
                X_train
            )

            X_test_processed = preprocessor.transform(
                X_test
            )

            # Mô hình mới; không ROS/SMOTE.
            model = dw_get_model()

            model.fit(
                X_train_processed,
                y_train
            )

            probability = model.predict_proba(
                X_test_processed
            )[:, 1]

            if probability.shape != (len(test_idx),):
                raise RuntimeError(
                    f"Nhánh {branch}: sai kích thước dự báo."
                )

            if (
                not np.isfinite(probability).all()
                or not (
                    (probability >= 0)
                    & (probability <= 1)
                ).all()
            ):
                raise RuntimeError(
                    f"Nhánh {branch}: xác suất không hợp lệ."
                )

            branch_prob[branch] = probability
            branch_auc[branch] = float(
                roc_auc_score(y_test, probability)
            )

            print(
                f"Nhánh {branch}: "
                f"AUC={branch_auc[branch]:.6f}",
                flush=True
            )

            del model, preprocessor
            del X_train_processed, X_test_processed
            del X_train, X_test
            gc.collect()

        # Hai nhánh nằm trên cùng ID/nhãn test.
        prediction_table = pd.DataFrame({
            "CUSTOMER_NUMBER": test_ids,
            "Y_TRUE": y_test,
            "PROB_A": branch_prob["A"],
            "PROB_B": branch_prob["B"]
        })

        prediction_table.to_csv(
            DW_RUN_DIR
            / f"predictions_seed_{split_seed}.csv",
            index=False
        )

        row = {
            "run": run_number,
            "split_seed": split_seed,
            "model_seed": DW_MODEL_SEED,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "positive_train": int(y_train.sum()),
            "positive_test": int(y_test.sum()),
            "AUC_A": branch_auc["A"],
            "AUC_B": branch_auc["B"],
            "D_window": (
                branch_auc["B"]
                - branch_auc["A"]
            ),
            "seconds": (
                time.perf_counter()
                - split_start
            )
        }

        DW_ROWS.append(row)

        # Lưu kết quả thật sau mỗi cặp A/B hoàn tất.
        pd.DataFrame(DW_ROWS).to_csv(
            DW_RUN_DIR / "Dwindow_per_split.csv",
            index=False
        )

        DW_METADATA["completed_splits"] = len(DW_ROWS)
        dw_save_metadata()

        print(
            f"D_window = B − A = {row['D_window']:.6f}",
            flush=True
        )

        print(
            f"Thời gian lần này: {row['seconds']:.1f} giây",
            flush=True
        )

except BaseException as error:
    DW_METADATA["status"] = "incomplete"
    DW_METADATA["error"] = repr(error)
    dw_save_metadata()

    print("\nCHƯA HOÀN TẤT")
    print("Số cặp A/B đã hoàn tất:", len(DW_ROWS))
    print("Kết quả từng cặp được lưu tại:", DW_RUN_DIR)

    if DW_ROWS:
        display(pd.DataFrame(DW_ROWS).round(6))

    raise


# ============================================================
# 8. TỔNG KẾT SAU KHI ĐỦ 20 LẦN
# ============================================================

DW_PER_SPLIT = pd.DataFrame(DW_ROWS)

if len(DW_PER_SPLIT) != DW_R:
    raise RuntimeError(
        "Chưa đủ 20 lần; chưa xuất tổng kết cho bản thảo."
    )

DW_DIFF = DW_PER_SPLIT["D_window"].to_numpy()

DW_Q025, DW_Q975 = np.quantile(
    DW_DIFF,
    [0.025, 0.975],
    method="linear"
)

DW_SUMMARY = pd.DataFrame([{
    "R_completed": len(DW_DIFF),
    "Dwindow_mean": float(DW_DIFF.mean()),
    "Dwindow_q025": float(DW_Q025),
    "Dwindow_q975": float(DW_Q975),
    "n_Dwindow_gt_0": int((DW_DIFF > 0).sum()),
    "AUC_A_mean": float(DW_PER_SPLIT["AUC_A"].mean()),
    "AUC_B_mean": float(DW_PER_SPLIT["AUC_B"].mean()),
    "percentile_method": "linear"
}])

DW_SUMMARY.to_csv(
    DW_RUN_DIR / "Dwindow_refit_summary.csv",
    index=False
)

DW_METADATA["status"] = "complete"
DW_METADATA["total_seconds"] = (
    time.perf_counter()
    - DW_TOTAL_START
)

dw_save_metadata()


# ============================================================
# 9. HIỂN THỊ KẾT QUẢ NGAY BÊN DƯỚI Ô CODE
# ============================================================

print("\n===== KẾT QUẢ CỦA CẢ 20 LẦN =====")

display(
    DW_PER_SPLIT[[
        "run",
        "split_seed",
        "model_seed",
        "n_train",
        "n_test",
        "positive_train",
        "positive_test",
        "AUC_A",
        "AUC_B",
        "D_window",
        "seconds"
    ]].round(6)
)

print("\n===== BẢNG TỔNG HỢP =====")
display(DW_SUMMARY.round(6))

DW_RESULT = DW_SUMMARY.iloc[0]

print("\n===== CÁC GIÁ TRỊ ĐỂ ĐIỀN BẢN THẢO =====")

print(
    f"R = {int(DW_RESULT['R_completed'])}"
)

print(
    "Mean D_window = "
    f"{DW_RESULT['Dwindow_mean']:.6f}"
)

print(
    "Empirical 2.5–97.5% percentile interval = "
    f"[{DW_RESULT['Dwindow_q025']:.6f}, "
    f"{DW_RESULT['Dwindow_q975']:.6f}]"
)

print(
    "D_window > 0 trong "
    f"{int(DW_RESULT['n_Dwindow_gt_0'])}/"
    f"{int(DW_RESULT['R_completed'])} lần chia."
)

print(
    "\nKhoảng trên là phân vị giữa các lần chia, "
    "không phải khoảng tin cậy 95% của trung bình."
)

print("\n===== TỆP ĐÃ LƯU =====")
print("Thư mục:", DW_RUN_DIR)
print("1. Dwindow_per_split.csv")
print("2. Dwindow_refit_summary.csv")
print("3. run_manifest.json")
print("4. ID train/test và dự báo A/B của từng seed")

print(
    "\nTổng thời gian: "
    f"{DW_METADATA['total_seconds'] / 60:.2f} phút"
)

Mounted at /content/drive

LOAD A
File: /content/drive/MyDrive/Fintect/final_landmark_60d/final_dataset_landmark_60d_no_auto_job.csv
Shape: (94453, 21)
Customers: 94,453
Positives: 4,374

LOAD B
File: /content/drive/MyDrive/Fintect/VIB_FLDC_60D.csv
Shape: (94453, 25)
Customers: 94,453
Positives: 4,374

===== KIỂM TRA A/B =====
Khách hàng A: 94,453
Khách hàng B: 94,453
Ca dương: 4,374
Target mismatch: 0
Cùng thứ tự ID: True
Đặc trưng mỗi nhánh: 18
Seed mô hình: 42
Seed chia dữ liệu: [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019]
Sampling: None

Thư mục kết quả: /content/drive/MyDrive/Fintect/VIB_DWINDOW_REFIT_20SEEDS/run_20261005T161147532827Z
Phiên bản thư viện: {'numpy': '2.1.3', 'pandas': '2.2.3', 'scikit-learn': '1.6.1', 'xgboost': '3.4.1'}

===== LẦN 1/20 — SPLIT SEED 1000 =====
Train=75,562; Test=18,891; Positive train=3,499; Positive test=875
Đang fit nhánh A...
Nhánh A: AUC=0.910954
Đang fit nhánh B...
Nhá

,run,split_seed,model_seed,n_train,n_test,positive_train,positive_test,AUC_A,AUC_B,D_window,seconds
0,1,1000,42,75562,18891,3499,875,0.910954,0.967169,0.056215,10.055650
1,2,1001,42,75562,18891,3499,875,0.908038,0.962123,0.054085,10.898578
2,3,1002,42,75562,18891,3499,875,0.909189,0.961497,0.052308,11.236610
3,4,1003,42,75562,18891,3499,875,0.904619,0.960374,0.055755,7.756856
4,5,1004,42,75562,18891,3499,875,0.904505,0.962098,0.057593,10.931044
5,6,1005,42,75562,18891,3499,875,0.907581,0.962860,0.055279,10.965364
6,7,1006,42,75562,18891,3499,875,0.911265,0.959346,0.048081,7.692781
7,8,1007,42,75562,18891,3499,875,0.912790,0.961558,0.048768,10.934267
8,9,1008,42,75562,18891,3499,875,0.908760,0.963934,0.055174,10.954297
9,10,1009,42,75562,18891,3499,875,0.908577,0.964102,0.055525,7.883592



===== BẢNG TỔNG HỢP =====


,R_completed,Dwindow_mean,Dwindow_q025,Dwindow_q975,n_Dwindow_gt_0,AUC_A_mean,AUC_B_mean,percentile_method
0,20,0.052975,0.046268,0.058488,20,0.909488,0.962463,linear



===== CÁC GIÁ TRỊ ĐỂ ĐIỀN BẢN THẢO =====
R = 20
Mean D_window = 0.052975
Empirical 2.5–97.5% percentile interval = [0.046268, 0.058488]
D_window > 0 trong 20/20 lần chia.

Khoảng trên là phân vị giữa các lần chia, không phải khoảng tin cậy 95% của trung bình.

===== TỆP ĐÃ LƯU =====
Thư mục: /content/drive/MyDrive/Fintect/VIB_DWINDOW_REFIT_20SEEDS/run_20261005T161147532827Z
1. Dwindow_per_split.csv
2. Dwindow_refit_summary.csv
3. run_manifest.json
4. ID train/test và dự báo A/B của từng seed

Tổng thời gian: 3.30 phút
